# Neural Amp Modeler ("Easy Mode" Trainer)
**Note**:
This notebook is meant to be used on [Google Colab](https://colab.research.google.com/github/sdatkinson/neural-amp-modeler/blob/main/bin/train/easy_colab.ipynb).

🔶**Before you run**🔶

Make sure to get a GPU! (From the upper-left menu, click Runtime->Change runtime type->Select "GPU" from the "Hardware accelerator dropdown menu)

## Step 1: Get data
* **Download the reamp signal.** Here: [v3_0_0.wav](https://drive.google.com/file/d/1Pgf8PdE0rKB1TD4TRPKbpNo1ByR3IOm9/view?usp=drive_link).
* **Reamp your gear.** Then reamp the gear you want to model using it. Save that reamp as "output.wav". *Note: Use 48kHz, 24-bit, mono.* For other sample rates, use [the CLI trainer](https://github.com/sdatkinson/neural-amp-modeler).
* **Upload your files.** Upload the input (DI) and output (amped) files you want to use by clicking the Folder icon on the left ⬅ and then clicking the upload icon or by dragging the files into the panel.

## Step 2: Train!
Configure your training run below, then hit the Play button to start training!

🕙NOTE: At default settings, training will take about 10 minutes.🕙

In [ ]:
import json
import re
import sys
import time
import traceback
import shutil
from functools import partial
from pathlib import Path

# =====================================================================
# 1. GOOGLE DRIVE ENTEGRASYONU VE DOSYA KONTROLÜ
# =====================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_DIR = Path("/content/drive/MyDrive/Namplifier/current")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    
    if (DRIVE_DIR / "input.wav").exists():
        !(cp "{DRIVE_DIR}/input.wav" .)
    if (DRIVE_DIR / "output.wav").exists():
        !(cp "{DRIVE_DIR}/output.wav" .)
    print("Google Drive senkronizasyonu başarılı.")
except Exception as e:
    print(f"Drive bağlantı uyarısı: {e}")
    DRIVE_DIR = Path(".")

# =====================================================================
# 2. NAM KURULUMU
# =====================================================================
try:
    import nam
except ImportError:
    print("NAM kuruluyor...")
    !if [ ! -d logs ]; then mkdir logs; fi
    !pip install neural-amp-modeler > logs/install.log 2>&1

from nam.train.colab import run
from nam.models.metadata import GearType, ToneType, UserMetadata

try:
    %load_ext tensorboard
except Exception:
    %reload_ext tensorboard

# =====================================================================
# 3. EĞİTİM VE PARAMETRE YAPILANDIRMASI
# =====================================================================
EPOCHS = 20  
architecture = "lite"
latency_samples = "auto"
fit_cab = False
ignore_checks = True

use_metadata = False
name = "Namplifier Capture"
modeled_by = "User"
gear_make = "Custom"
gear_model = "Amp"
gear_type = "amp"
tone_type = "clean"

PROGRESS_PATH = DRIVE_DIR / "progress.json"
LOG_PATH = DRIVE_DIR / "training.log"

# Terminaldeki ANSI renk ve imleç kodlarını temizleyen regex
_ANSI_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')

# Gelişmiş Epoch ve İlerleme Yakalayıcı Regex
# Örn: "Epoch 3: 15%", "Epoch 3/20: 15%", "Epoch 3: 15/100" durumlarını kapsar
_EPOCH_RE = re.compile(r"Epoch\s+(\d+)(?:/(\d+))?[:\s]+(\d+)%", re.IGNORECASE)

# =====================================================================
# 4. YARDIMCI FONKSİYONLAR VE STREAM LOG YAKALAYICI
# =====================================================================
def _now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def write_progress(status, percent=None, epoch=None, total_epochs=EPOCHS, message=""):
    payload = {
        "status": status,          # "queued" | "training" | "done" | "error"
        "percent": percent,
        "epoch": epoch,
        "total_epochs": total_epochs,
        "message": message,
        "updated_at": _now(),
    }
    PROGRESS_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2))

def log_line(text):
    line = f"[{_now()}] {text}"
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

class _TeeStream:
    def __init__(self, real_stream):
        self.real_stream = real_stream
        self._buf = ""

    def write(self, data):
        self.real_stream.write(data)
        self._buf += data
        
        # Satır sonu (\n) veya carriage return (\r) geldikçe parçala
        while "\n" in self._buf or "\r" in self._buf:
            idx_n = self._buf.find("\n")
            idx_r = self._buf.find("\r")
            
            if idx_n != -1 and (idx_r == -1 or idx_n < idx_r):
                sep = "\n"
            else:
                sep = "\r"
                
            line, self._buf = self._buf.split(sep, 1)
            
            # ANSI escape karakterlerini temizle (tqdm progress bar görsellerini süzmek için)
            clean_line = _ANSI_RE.sub('', line).strip()
            if not clean_line:
                continue
                
            log_line(clean_line)
            
            # Epoch durumunu regex ile yakala
            m = _EPOCH_RE.search(clean_line)
            if m:
                epoch_val = int(m.group(1))
                pct_in_epoch = int(m.group(3))
                
                # Lightning varsayılan olarak Epoch sayımına 0'dan başlar (0-19)
                current_epoch_display = epoch_val + 1 if epoch_val < EPOCHS else epoch_val
                
                # Genel yüzde hesabı (%1 - %99 arası)
                overall_pct = min(99, max(1, round(((epoch_val + (pct_in_epoch / 100.0)) / EPOCHS) * 100)))
                
                write_progress(
                    status="training", 
                    percent=overall_pct, 
                    epoch=current_epoch_display, 
                    total_epochs=EPOCHS, 
                    message=f"Epoch {current_epoch_display}/{EPOCHS} (%{pct_in_epoch})"
                )

    def flush(self):
        self.real_stream.flush()

def _verbose_enum(E, val):
    try:
        return E(val)
    except ValueError as e:
        raise ValueError(f"{e}\nGeçerli seçenekler: " + ", ".join(list(x.value for x in E)))

def _parse_latency(ls: str):
    if str(ls).lower() == "auto":
        return None
    try:
        return int(ls)
    except ValueError as e:
        raise ValueError(f"Geçersiz latency: {ls}")

# =====================================================================
# 5. EĞİTİMİ BAŞLATMA VE GARANTİLİ DOSYA KOPYALAMA
# =====================================================================
user_metadata = None if not use_metadata else UserMetadata(
    name=name,
    modeled_by=modeled_by,
    gear_make=gear_make,
    gear_model=gear_model,
    gear_type=_verbose_enum(GearType, gear_type.lower()),
    tone_type=_verbose_enum(ToneType, tone_type.lower())
)

LOG_PATH.write_text("")
write_progress("queued", percent=0, epoch=0, message="Eğitim kuyruğuna alındı.")

_stdout, _stderr = sys.stdout, sys.stderr
sys.stdout, sys.stderr = _TeeStream(_stdout), _TeeStream(_stderr)

try:
    write_progress("training", percent=1, epoch=0, message="Eğitim başlatılıyor...")
    
    # NAM Eğitimi
    run(
        epochs=EPOCHS,
        delay=_parse_latency(latency_samples),
        user_metadata=user_metadata,
        ignore_checks=ignore_checks,
    )
    
    # --- MODEL DOSYASINI DRIVE'A AKTARMA ---
    export_dir = Path("exported_models")
    nam_files = list(export_dir.glob("*.nam")) if export_dir.exists() else []
    
    if not nam_files:
        nam_files = list(Path(".").glob("*.nam"))
        
    if nam_files:
        latest_nam = max(nam_files, key=lambda p: p.stat().st_mtime)
        target_path = DRIVE_DIR / "model.nam"
        shutil.copy(latest_nam, target_path)
        log_line(f"Model başarıyla Drive'a aktarıldı: {target_path}")
        write_progress("done", percent=100, epoch=EPOCHS, message="Model eğitimi tamamlandı ve Drive'a aktarıldı.")
    else:
        raise FileNotFoundError("Eğitim bitti ancak üretilen .nam dosyası sistemde bulunamadı!")

except Exception as e:
    write_progress("error", message=f"{type(e).__name__}: {e}")
    log_line("ERROR: " + traceback.format_exc())
    raise
finally:
    sys.stdout, sys.stderr = _stdout, _stderr

## Step 3: Check the results and download your model
We're done!

Have a look at the plot above to see how your model compares to the real gear you're modeling.
Hopefully it looks good!
Go to the file browser on the left panel ⬅ and download `model.nam` from the `exported_model` directory (you may need to hit the refresh button).

Additionally, if you want to continue to train this model later you can download the lightning model artifacts from `lightning_logs`. If not, that's fine too.

# 🎸 **ENJOY!** 🎸